# Stage 5 handoff: frozen Faster R-CNN regions + LSTM attention

This is a **starter for laptop B's Codex**, not a completed full-training notebook. Data loading, exact metrics, the shared masked LSTM, region extraction/cache and masked batch operations are included here. The recipient must integrate the full evaluation, preflight, checkpoint/resume and training loop using the supplied read-only stage-3 reference. See `PROMPT_FOR_CODEX_B.txt`.

Copy this file to **FasterRCNN_Stage5.ipynb** before editing. Keep this starter and the delivered data/protocol unchanged. The final B notebook should contain all runtime code; it can later be merged into the single submission notebook.

Only one new run is planned: **frozen Faster R-CNN ResNet50-FPN COCO_V1 + 3-layer additive-attention LSTM**. The caption decoder starts from scratch. This is inspired by bottom-up attention, not a reproduction of the original ResNet101/Visual Genome model.

The caption test is held out from caption training, but comes from COCO train2017 used by detector pretraining. It is not a clean image holdout for this detector. **No test metrics in this stage.** Compare validation with the frozen stage-3 grid LSTM; differences also include pretraining, resolution and feature dimension.


## 1. Imports and settings

All runtime code is in this notebook. The default review mode checks saved preparation and metrics without downloading weights, extracting features or training.

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import json
import hashlib
import pickle
from pathlib import Path
from collections import Counter, defaultdict
import random
import numpy as np
import torch
import re
import inspect
import unicodedata
import importlib.metadata as package_metadata
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.bleu.bleu_scorer import BleuScorer
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.cider.cider_scorer import CiderScorer
from torch import nn
import torchvision
from torchvision import transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.ops import nms
from PIL import Image, ImageOps
from datetime import datetime, timezone
import io
import os
import shutil
import time
from functools import partial
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset, Subset
import gc
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
BUNDLE_DIR = Path.cwd()
DATA_ROOT = BUNDLE_DIR / "data" / "coco2017"
RAW_IMAGE_ROOT = DATA_ROOT
# If JPEGs are stored elsewhere, set RAW_IMAGE_ROOT to the folder containing train2017 and val2017.
CACHE_DIR = BUNDLE_DIR / "cache" / "features" / "fasterrcnn_regions_v1"
RESULTS_DIR = BUNDLE_DIR / "experiments" / "fasterrcnn_regions_v1"
RUN_DETECTOR_SMOKE = False
ALLOW_WEIGHT_DOWNLOAD = False
EXTRACT_REGION_FEATURES = False
SEED = 42
torch.set_num_threads(4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
protocol = json.loads((BUNDLE_DIR / "protocol.json").read_text(encoding="utf-8"))
print("Device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")
print("Starter mode: full training must be integrated by laptop B's Codex.")


In [ ]:
def load_region_prepared_data(project_dir, data_root=None):
    project_dir = Path(project_dir)
    data_root = Path(data_root) if data_root is not None else project_dir / "data" / "coco2017"
    for source in ["train2017", "val2017"]:
        annotation_path = data_root / "annotations" / f"captions_{source}.json"
        if not annotation_path.is_file():
            raise FileNotFoundError(
                f"Missing {annotation_path}. Extract the shared training-data bundle beside "
                "Stage5_Starter.ipynb, or set DATA_ROOT to the bundled annotation folder. "
                "Raw train2017/val2017 JPEGs are required for region extraction."
            )
    cache = project_dir / "cache"
    required = [
        "splits.json",
        "prepared.pkl",
        "text_report.json",
        "vocab.json",
        "image_audit.json",
        "test_captions.json",
    ]
    missing = [name for name in required if not (cache / name).is_file()]
    if missing:
        raise FileNotFoundError(
            f"Missing prepared files: {missing}. Extract the shared training-data bundle into PROJECT_DIR. "
            "Do not recreate the split or vocabulary for this comparison."
        )

    # Load saved preparation results, then verify their versions agree.
    split_bytes = (cache / "splits.json").read_bytes()
    vocab_bytes = (cache / "vocab.json").read_bytes()
    prepared_bytes = (cache / "prepared.pkl").read_bytes()
    split_file = json.loads(split_bytes)
    vocab = json.loads(vocab_bytes)
    report = json.loads((cache / "text_report.json").read_text(encoding="utf-8"))
    audit = json.loads((cache / "image_audit.json").read_text(encoding="utf-8"))

    prepared_sha = hashlib.sha256(prepared_bytes).hexdigest()
    assert (
        prepared_sha == report["cache_sha256"]
    ), "Prepared caption cache changed. Rerun preparation."
    prepared = pickle.loads(
        prepared_bytes
    )  # This is our own cache, checked against its saved hash.
    del prepared_bytes

    split_ids = split_file["ids"]
    sources = split_file["feature_sources"]
    assert set(split_ids) == {"train", "val", "test"}
    assert sources == {
        "train": "train2017",
        "val": "val2017",
        "test": "train2017",
    }, "Use the saved labeled split, not official test2017."
    payload = {"config": split_file["config"], "ids": split_ids, "feature_sources": sources}
    split_id = hashlib.sha256(
        json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()
    ).hexdigest()
    assert split_id == split_file["split_id"] == prepared["config"]["split_id"] == vocab["split_id"]

    source_id = split_file["config"]["source_id"]
    assert source_id == prepared["config"]["source_id"] == vocab["source_id"] == audit["source_id"]
    assert prepared["config"] == report["config"]
    assert audit["complete"] and not audit["image_errors"] and not audit["metadata_errors"]
    assert (
        prepared["token_to_id"] == vocab["token_to_id"]
        and prepared["id_to_token"] == vocab["id_to_token"]
    )
    assert vocab["token_to_id"] == {word: index for index, word in enumerate(vocab["id_to_token"])}

    ids = {
        name: vocab["token_to_id"][token]
        for name, token in [
            ("pad", "<pad>"),
            ("unk", "<unk>"),
            ("start", "<start>"),
            ("end", "<end>"),
        ]
    }
    assert len(set(ids.values())) == 4

    # One photograph and its exact duplicates must stay in one labeled split.
    sets = {name: set(values) for name, values in split_ids.items()}
    for name, values in split_ids.items():
        assert values == sorted(sets[name]) and values, f"Invalid or repeated IDs in {name}."
    for left, right in [("train", "val"), ("train", "test"), ("val", "test")]:
        assert not sets[left] & sets[right], f"Image overlap: {left}/{right}."
    assert len(sets["test"]) == split_file["config"]["holdout_size"]

    owners = {
        (sources[name], image_id): name for name, values in sets.items() for image_id in values
    }
    for group in audit["pixel_duplicates"] + audit["file_duplicates"]:
        assigned = {owners[tuple(item)] for item in group if tuple(item) in owners}
        assert len(assigned) <= 1, "Exact duplicate photos cross the labeled splits."

    # Keep the original human references for evaluation.
    raw_by_source = {}
    source_image_ids = {}
    annotation_hashes = {}

    for source in ["train2017", "val2017"]:
        annotation_bytes = (data_root / "annotations" / f"captions_{source}.json").read_bytes()
        annotation_hashes[source] = hashlib.sha256(annotation_bytes).hexdigest()
        annotation = json.loads(annotation_bytes)
        source_image_ids[source] = sorted(row["id"] for row in annotation["images"])
        assert len(source_image_ids[source]) == len(set(source_image_ids[source]))

        raw_by_source[source] = defaultdict(list)
        for row in annotation["annotations"]:
            assert (
                isinstance(row["caption"], str) and row["caption"].strip()
            ), "Empty human reference."
            raw_by_source[source][row["image_id"]].append(row["caption"])
        assert set(raw_by_source[source]) == set(
            source_image_ids[source]
        ), "Missing human descriptions."

        if source == "train2017":
            saved_test = json.loads((cache / "test_captions.json").read_text(encoding="utf-8"))
            assert saved_test["split_id"] == split_id
            expected_test = [
                row for row in annotation["annotations"] if row["image_id"] in sets["test"]
            ]
            assert (
                saved_test["annotations"] == expected_test
            ), "Saved test references differ from the original human captions."
        del annotation_bytes, annotation

    assert sets["train"] | sets["test"] == set(source_image_ids["train2017"])
    assert sets["val"] == set(source_image_ids["val2017"])

    references = {
        name: {image_id: raw_by_source[sources[name]][image_id] for image_id in values}
        for name, values in split_ids.items()
    }
    tokens = {name: prepared[f"{name}_tokens"] for name in split_ids}
    for name, values in tokens.items():
        assert sorted(values) == split_ids[name]
        assert all(
            rows and all(row for row in rows) for rows in values.values()
        ), "Empty tokenized caption."
        if name != "train":
            assert all(
                len(rows) == len(references[name][image_id]) for image_id, rows in values.items()
            ), "Evaluation references were dropped."

    # Test and validation captions must not contribute to the vocabulary.
    train_counts = Counter(
        token for rows in tokens["train"].values() for row in rows for token in row
    )
    assert train_counts == prepared["train_counts"]
    settings = prepared["config"]["settings"]
    regular = [
        token
        for token, count in sorted(train_counts.items(), key=lambda row: (-row[1], row[0]))
        if count >= settings["min_frequency"]
    ][: settings["max_vocab_tokens"]]
    assert (
        vocab["id_to_token"] == settings["special_tokens"] + regular
    ), "Vocabulary must be fitted on training captions only."

    data = {
        "project_dir": project_dir,
        "data_root": data_root,
        "split_ids": split_ids,
        "feature_sources": sources,
        "tokens": tokens,
        "references": references,
        "source_image_ids": source_image_ids,
        "token_to_id": vocab["token_to_id"],
        "id_to_token": vocab["id_to_token"],
        "ids": ids,
        "max_content_tokens": prepared["config"]["max_content_tokens"],
        "text_settings": settings,
        "provenance": {
            "source_id": source_id,
            "split_id": split_id,
            "prepared_sha256": prepared_sha,
            "vocab_sha256": hashlib.sha256(vocab_bytes).hexdigest(),
            "splits_sha256": hashlib.sha256(split_bytes).hexdigest(),
            "annotations_sha256": annotation_hashes,
        },
    }

    print(
        "Verified disjoint labeled splits:",
        {name: len(values) for name, values in split_ids.items()},
    )
    print("Vocabulary:", len(data["id_to_token"]), "| special IDs:", ids)
    return data

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

def training_order(size, seed, epoch):
    return torch.randperm(size, generator=torch.Generator().manual_seed(seed + epoch)).tolist()

In [ ]:
data = load_region_prepared_data(BUNDLE_DIR, DATA_ROOT)
assert data["provenance"] == protocol["data"]
assert {name: len(ids) for name, ids in data["split_ids"].items()} == {"train":113287,"val":5000,"test":5000}
print("Saved split, vocabulary, preparation and raw human references match the shared protocol.")


## 2. Shared decoder

Features have shape **[B,36,1024]**, with a boolean **[B,36]** mask (`True=valid`). The trainable projection produces [B,36,512]. A masked mean is the first visual LSTM step, starting from zero hidden/cell states. The visual output has no word target. At each word, additive attention uses the previous top hidden state; context and word embedding are concatenated and projected to 512.

The recurrent block has **3 layers, width512, built-in dropout0**. Separate dropout0.3 follows word embeddings and precedes the vocabulary output. No extra layer normalization or region position/box embeddings are added. Coordinates, classes and scores are saved for audit, not fed to the decoder.

The same decoder mathematics were verified against stage3 on 49 valid2048-D regions. Only the feature projection input and visual mask differ. Stage3 weights are not loaded into the new1024-input model.


In [ ]:
class CaptionRegionLSTM(nn.Module):
    """The spatial LSTM decoder with a variable number of valid visual regions."""

    def __init__(
        self,
        feature_dim,
        vocab_size,
        pad_id,
        embedding_size=512,
        hidden_size=512,
        num_layers=3,
        dropout=0.3,
    ):
        super().__init__()
        if embedding_size != hidden_size:
            raise ValueError("This model uses equal embedding and hidden sizes.")
        if num_layers < 1:
            raise ValueError("Use at least one recurrent layer.")

        self.decoder_type = "lstm_attention"
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.pad_id = pad_id

        # Keep the original modules and construction order for dense-grid comparisons.
        self.image_projection = nn.Linear(feature_dim, hidden_size)
        self.embedding = nn.Embedding(vocab_size, embedding_size, padding_idx=pad_id)
        self.embedding_dropout = nn.Dropout(dropout)
        self.attention_visual = nn.Linear(hidden_size, hidden_size)
        self.attention_hidden = nn.Linear(hidden_size, hidden_size, bias=False)
        self.attention_score = nn.Linear(hidden_size, 1, bias=False)
        self.input_fusion = nn.Linear(embedding_size + hidden_size, embedding_size)
        self.recurrent = nn.LSTM(
            embedding_size,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=0,
        )
        self.output_dropout = nn.Dropout(dropout)
        self.output = nn.Linear(hidden_size, vocab_size)

    def prepare_image(self, features, visual_mask=None):
        if features.ndim != 3 or features.shape[1] < 1:
            raise ValueError("Expected visual features with shape [batch, K, feature_dim], K >= 1.")
        if features.shape[2] != self.image_projection.in_features:
            raise ValueError("The cached feature dimension does not match the model.")
        if visual_mask is not None:
            if not isinstance(visual_mask, torch.Tensor) or visual_mask.dtype != torch.bool:
                raise TypeError("visual_mask must be a boolean tensor; True means valid.")
            if visual_mask.shape != features.shape[:2]:
                raise ValueError("visual_mask must have shape [batch, K].")
            visual_mask = visual_mask.to(features.device)
            if not visual_mask.any(dim=1).all():
                raise ValueError("Each image must have at least one valid visual region.")
            if visual_mask.all():
                # Preserve the original dense-grid calculation exactly.
                visual_mask = None

        if visual_mask is not None:
            # Remove padded garbage, including NaNs, before any learned operation.
            features = features.masked_fill(~visual_mask.unsqueeze(-1), 0)
        visual = self.image_projection(features)
        if visual_mask is not None:
            visual = visual.masked_fill(~visual_mask.unsqueeze(-1), 0)
        return visual, visual_mask

    def initial_state(self, visual, visual_mask=None):
        hidden = visual.new_zeros(self.num_layers, len(visual), self.hidden_size)
        initial = (hidden, torch.zeros_like(hidden))
        if visual_mask is None:
            image_mean = visual.mean(dim=1)
        else:
            valid_values = visual.masked_fill(~visual_mask.unsqueeze(-1), 0)
            valid_count = visual_mask.sum(dim=1, keepdim=True).to(visual.dtype)
            image_mean = valid_values.sum(dim=1) / valid_count
        # Process the mean image separately; this step has no word target.
        _, state = self.recurrent(image_mean.unsqueeze(1), initial)
        return state

    def image_state(self, features, visual_mask=None):
        visual, visual_mask = self.prepare_image(features, visual_mask)
        return self.initial_state(visual, visual_mask)

    def word_step(self, token_ids, visual, visual_keys, state, visual_mask=None):
        query = self.attention_hidden(state[0][-1]).unsqueeze(1)
        scores = self.attention_score(torch.tanh(visual_keys + query)).squeeze(-1)
        if visual_mask is not None:
            scores = scores.masked_fill(~visual_mask, -torch.inf)
        attention = torch.softmax(scores, dim=1)
        context = torch.sum(visual * attention.unsqueeze(-1), dim=1)

        word = self.embedding_dropout(self.embedding(token_ids))
        recurrent_input = self.input_fusion(torch.cat([word, context], dim=-1))
        output, state = self.recurrent(recurrent_input.unsqueeze(1), state)
        logits = self.output(self.output_dropout(output[:, 0]))
        return logits, state, attention

    def forward_with_attention(self, features, input_ids, visual_mask=None):
        if input_ids.ndim != 2 or input_ids.shape[1] < 1 or len(features) != len(input_ids):
            raise ValueError("Each image needs a nonempty input token sequence.")
        visual, visual_mask = self.prepare_image(features, visual_mask)
        visual_keys = self.attention_visual(visual)
        state = self.initial_state(visual, visual_mask)
        logits = []
        attention = []

        # Input: <start> and previous words; outputs predict the next words.
        for token_ids in input_ids.unbind(dim=1):
            next_logits, state, weights = self.word_step(
                token_ids, visual, visual_keys, state, visual_mask
            )
            logits.append(next_logits)
            attention.append(weights)
        return torch.stack(logits, dim=1), torch.stack(attention, dim=1)

    def forward(self, features, input_ids, visual_mask=None):
        logits, _ = self.forward_with_attention(features, input_ids, visual_mask)
        return logits

    @torch.no_grad()
    def greedy(self, features, start_id, end_id, max_steps=33, visual_mask=None):
        if max_steps < 1:
            raise ValueError("max_steps must be positive.")
        was_training = self.training
        self.eval()
        try:
            visual, visual_mask = self.prepare_image(features, visual_mask)
            visual_keys = self.attention_visual(visual)
            state = self.initial_state(visual, visual_mask)
            current = torch.full(
                (len(features),), start_id, dtype=torch.long, device=features.device
            )
            finished = torch.zeros(len(features), dtype=torch.bool, device=features.device)
            predictions = [[] for _ in range(len(features))]
            for _ in range(max_steps):
                logits, state, _ = self.word_step(
                    current, visual, visual_keys, state, visual_mask
                )
                logits[:, self.pad_id] = -torch.inf
                logits[:, start_id] = -torch.inf
                next_ids = logits.argmax(dim=-1)
                for index, (token_id, done) in enumerate(zip(next_ids.tolist(), finished.tolist())):
                    if not done and token_id != end_id:
                        predictions[index].append(token_id)
                finished |= next_ids.eq(end_id)
                if finished.all():
                    break
                current = next_ids
            return predictions
        finally:
            self.train(was_training)

In [ ]:
seed_everything(SEED)
example = CaptionRegionLSTM(1024,len(data["id_to_token"]),data["ids"]["pad"])
parameter_count = sum(p.numel() for p in example.parameters() if p.requires_grad)
assert parameter_count == protocol["architecture"]["trainable_parameters"]
print("Trainable caption parameters:", parameter_count)
del example
example_words = ["a","cat"]
example_caption = [data["ids"]["start"]] + [data["token_to_id"].get(word,data["ids"]["unk"]) for word in example_words] + [data["ids"]["end"]]
print("Teacher input:", [data["id_to_token"][i] for i in example_caption[:-1]])
print("Word targets:", [data["id_to_token"][i] for i in example_caption[1:]])


## 3. Frozen detector and durable cache

Use original train2017/val2017 JPEGs, EXIF-transposed RGB tensors in[0,1]. Faster R-CNN performs its own normalization and resizing (short side800, long side at most1333). Do not reuse224-square classification tensors.

Extract final detections, keep scores>=0.2, apply additional class-agnostic NMS0.5 and retain at most36. Re-run ROI pooling and the1024-D box head on these final boxes to keep vectors aligned. An empty selection uses one full-image ROI and records a fallback flag. Boxes are saved in original image coordinates.

Compute detector features in float32 with frozen weights and eval mode; store vectors in float16. Initial extraction batch is **1**, independent of caption training batch64. The complete feature array needs about **8.47GiB**, plus metadata/checkpoints and original JPEGs. The source train cache includes held-out IDs but no caption fitting or test evaluation. Save cumulative extraction time, weight checksum, versions, counts and fallback statistics. A completed source is reused; an interrupted one resumes committed progress.


In [ ]:
REGION_TORCH_PIN = "2.13.0"
REGION_TORCHVISION_PIN = "0.28.0"
REGION_WEIGHT_NAME = "FasterRCNN_ResNet50_FPN_Weights.COCO_V1"
REGION_CACHE_VERSION = "fasterrcnn_resnet50_fpn_coco_v1_regions_v1"
REGION_DIM = 1024


def region_feature_contract(max_regions=36, score_threshold=0.2, nms_iou=0.5):
    return {
        "version": REGION_CACHE_VERSION,
        "detector": "torchvision.models.detection.fasterrcnn_resnet50_fpn",
        "weights": REGION_WEIGHT_NAME,
        "torch": REGION_TORCH_PIN,
        "torchvision": REGION_TORCHVISION_PIN,
        "input": "EXIF-transposed RGB; ToTensor float32 in [0,1]; original aspect ratio and size",
        "detector_transform": {
            "min_size": 800, "max_size": 1333, "size_divisible": 32,
            "image_mean": [0.485, 0.456, 0.406], "image_std": [0.229, 0.224, 0.225],
        },
        "compute_dtype": "float32; autocast disabled",
        "cnn_frozen": True,
        "detector_eval": True,
        "detector_score_threshold": 0.05,
        "detector_class_nms_iou": 0.5,
        "detector_max_detections": 100,
        "rpn_pre_nms_top_n_test": 1000,
        "rpn_post_nms_top_n_test": 1000,
        "rpn_nms_iou": 0.7,
        "selection": "Final class-NMS detections, score filter, class-agnostic NMS, descending score, top K",
        "score_threshold": float(score_threshold),
        "class_agnostic_nms_iou": float(nms_iou),
        "max_regions": int(max_regions),
        "feature_dimension": REGION_DIM,
        "feature_layer": "box_roi_pool + TwoMLPHead.fc6/ReLU/fc7/ReLU, rerun on selected FINAL boxes",
        "feature_storage_dtype": "float16 after finite/overflow validation; compute and returned features remain float32",
        "boxes": "xyxy in original EXIF-transposed RGB pixel coordinates; zero padding",
        "boxes_normalized": "xyxy divided by original [width,height,width,height]; reporting only",
        "mask": "bool; True means a real selected region, including the full-image fallback",
        "padding": "zero features/boxes/scores, label=-1, mask=False",
        "fallback": "If no boxes remain, ROI-pool the full transformed image; valid count=1, score=0, label=-1, fallback=True",
        "decoder_inputs": "visual features and valid-region mask only; no predicted class labels, scores or object words",
        "test_exposure_note": "COCO detector pretraining may include train2017 images used by the captioning holdout; do not claim an unseen-image encoder test.",
    }


def build_frozen_region_detector(device="cpu", allow_download=False):
    """No model is constructed or downloaded until this function is explicitly called."""
    if torch.__version__.split("+")[0] != REGION_TORCH_PIN or torchvision.__version__.split("+")[0] != REGION_TORCHVISION_PIN:
        raise RuntimeError(f"Use torch=={REGION_TORCH_PIN} and torchvision=={REGION_TORCHVISION_PIN} for this pinned region contract.")
    weights = FasterRCNN_ResNet50_FPN_Weights.COCO_V1
    filename = weights.url.rsplit("/", 1)[-1]
    checkpoint = Path(torch.hub.get_dir()) / "checkpoints" / filename
    if not checkpoint.is_file() and not allow_download:
        raise FileNotFoundError(f"Missing {checkpoint}. Downloading is disabled; set allow_download=True only when ready to obtain the pretrained detector.")
    if checkpoint.is_file():
        digest = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
        assert digest.startswith("258fb6c6"), "Cached detector weights do not match COCO_V1."
    detector = fasterrcnn_resnet50_fpn(
        weights=weights, weights_backbone=None, progress=True,
        min_size=800, max_size=1333,
        box_score_thresh=0.05, box_nms_thresh=0.5, box_detections_per_img=100,
        rpn_pre_nms_top_n_test=1000, rpn_post_nms_top_n_test=1000, rpn_nms_thresh=0.7,
    )
    detector = detector.to(device=torch.device(device), dtype=torch.float32).requires_grad_(False).eval()
    assert detector.roi_heads.box_head.fc7.out_features == REGION_DIM
    digest = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
    assert digest.startswith("258fb6c6")
    detector.region_feature_metadata = {**region_feature_contract(), "weights_sha256": digest,
                                        "device": str(torch.device(device))}
    return detector


def load_region_image(path):
    """Do not apply the 224x224 classification resize, padding or normalization here."""
    with Image.open(path) as image:
        return transforms.ToTensor()(ImageOps.exif_transpose(image).convert("RGB"))


def _select_region_boxes(detection, transformed_size, max_regions, score_threshold, nms_iou):
    boxes, scores, labels = detection["boxes"], detection["scores"], detection["labels"]
    assert boxes.ndim == 2 and boxes.shape[1] == 4
    assert scores.shape == labels.shape == (len(boxes),)
    assert torch.isfinite(boxes).all() and torch.isfinite(scores).all()
    assert torch.all((scores >= 0) & (scores <= 1))
    if len(boxes):
        assert torch.all(boxes[:, 2:] > boxes[:, :2]) and torch.all(labels > 0)
    keep = scores >= score_threshold
    boxes, scores, labels = boxes[keep], scores[keep], labels[keep]
    if len(boxes):
        order = torch.argsort(scores, descending=True, stable=True)
        boxes, scores, labels = boxes[order], scores[order], labels[order]
        keep = nms(boxes, scores, nms_iou)[:max_regions]
        return {"boxes": boxes[keep], "scores": scores[keep], "labels": labels[keep], "fallback": False}
    height, width = transformed_size
    return {
        "boxes": boxes.new_tensor([[0, 0, width, height]]),
        "scores": scores.new_zeros(1),
        "labels": torch.full((1,), -1, dtype=torch.long, device=boxes.device),
        "fallback": True,
    }


@torch.no_grad()
def extract_region_batch(detector, images, max_regions=36, score_threshold=0.2, nms_iou=0.5):
    """Return one CPU record per image; no files, caches or downloads are written."""
    if not isinstance(max_regions, int) or isinstance(max_regions, bool) or max_regions < 1:
        raise ValueError("max_regions must be a positive integer.")
    if not 0 <= score_threshold <= 1 or not 0 <= nms_iou <= 1:
        raise ValueError("score_threshold and nms_iou must be in [0,1].")
    if not isinstance(images, (list, tuple)) or not images:
        raise ValueError("Pass a nonempty list of original RGB image tensors.")
    if detector.training or any(parameter.requires_grad for parameter in detector.parameters()):
        raise ValueError("The detector must be frozen and in eval() mode.")
    device = next(detector.parameters()).device
    original_sizes = []
    prepared_images = []
    for image in images:
        if image.ndim != 3 or image.shape[0] != 3 or min(image.shape[-2:]) < 1:
            raise ValueError("Each image must have shape [3,H,W].")
        if not image.is_floating_point() or not torch.isfinite(image).all() or image.min() < 0 or image.max() > 1:
            raise ValueError("Use unnormalized RGB float tensors in [0,1].")
        original_sizes.append(tuple(image.shape[-2:]))
        prepared_images.append(image.to(device=device, dtype=torch.float32))

    # All detector/ROI computation stays float32, even inside a caller's AMP context.
    with torch.autocast(device_type=device.type, enabled=False):
        transformed, _ = detector.transform(prepared_images, None)
        fpn_features = detector.backbone(transformed.tensors)
        if isinstance(fpn_features, torch.Tensor):
            fpn_features = {"0": fpn_features}
        proposals, _ = detector.rpn(transformed, fpn_features, None)
        detections, _ = detector.roi_heads(fpn_features, proposals, transformed.image_sizes, None)
        assert len(detections) == len(images)
        selected = [_select_region_boxes(detection, size, max_regions, score_threshold, nms_iou)
                    for detection, size in zip(detections, transformed.image_sizes)]
        selected_boxes = [item["boxes"] for item in selected]

        # NMS/regression changes the proposal rows. Re-pool FINAL selected boxes;
        # never index the earlier proposal feature matrix using final detection IDs.
        roi_maps = detector.roi_heads.box_roi_pool(fpn_features, selected_boxes, transformed.image_sizes)
        roi_vectors = detector.roi_heads.box_head(roi_maps)
        counts = [len(boxes) for boxes in selected_boxes]
        assert roi_vectors.shape == (sum(counts), REGION_DIM) and roi_vectors.dtype == torch.float32
        assert torch.isfinite(roi_vectors).all()
        grouped_vectors = roi_vectors.split(counts, dim=0)

    records = []
    for vectors, selection, original_size, transformed_size in zip(grouped_vectors, selected, original_sizes, transformed.image_sizes):
        count = len(vectors)
        original_height, original_width = original_size
        transformed_height, transformed_width = transformed_size
        scale = selection["boxes"].new_tensor([original_width / transformed_width, original_height / transformed_height,
                                              original_width / transformed_width, original_height / transformed_height])
        boxes = selection["boxes"] * scale  # Exactly one transformed -> original rescaling.
        boxes[:, 0::2].clamp_(min=0, max=original_width)
        boxes[:, 1::2].clamp_(min=0, max=original_height)
        normalized = boxes / boxes.new_tensor([original_width, original_height, original_width, original_height])
        record = {
            "features": torch.zeros(max_regions, REGION_DIM, dtype=torch.float32),
            "boxes": torch.zeros(max_regions, 4, dtype=torch.float32),
            "boxes_normalized": torch.zeros(max_regions, 4, dtype=torch.float32),
            "scores": torch.zeros(max_regions, dtype=torch.float32),
            "labels": torch.full((max_regions,), -1, dtype=torch.long),
            "mask": torch.zeros(max_regions, dtype=torch.bool),
            "num_regions": count, "fallback": selection["fallback"],
            "original_size": original_size, "transformed_size": tuple(transformed_size),
        }
        record["features"][:count] = vectors.cpu()
        record["boxes"][:count] = boxes.cpu()
        record["boxes_normalized"][:count] = normalized.cpu()
        record["scores"][:count] = selection["scores"].cpu()
        record["labels"][:count] = selection["labels"].cpu()
        record["mask"][:count] = True
        assert 1 <= count <= max_regions and int(record["mask"].sum()) == count
        assert torch.isfinite(record["features"].to(torch.float16)).all(), "ROI features overflow float16 storage."
        records.append(record)
    return records

In [ ]:
REGION_SOURCES = ("train2017", "val2017")


def _region_json(path, value):
    temporary = path.with_suffix(".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2)
        handle.flush()
        os.fsync(handle.fileno())
    for attempt in range(10):
        try:
            temporary.replace(path)
            return
        except PermissionError:
            if attempt == 9:
                raise
            time.sleep(0.2)


def _region_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def _region_file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(4 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _region_shapes(count, max_regions):
    return {
        "features": ((count, max_regions, REGION_DIM), np.dtype("float16")),
        "mask": ((count, max_regions), np.dtype("bool")),
        "num_regions": ((count,), np.dtype("uint8")),
        "boxes": ((count, max_regions, 4), np.dtype("float32")),
        "scores": ((count, max_regions), np.dtype("float32")),
        "labels": ((count, max_regions), np.dtype("int64")),
        "fallback": ((count,), np.dtype("bool")),
    }


def _region_file_bytes(shape, dtype):
    header = io.BytesIO()
    np.lib.format.write_array_header_1_0(header, {"descr": np.lib.format.dtype_to_descr(dtype),
                                               "fortran_order": False, "shape": shape})
    return len(header.getvalue()) + int(np.prod(shape)) * dtype.itemsize


def _region_paths(cache_dir, source, shapes):
    directory = Path(cache_dir)
    return {
        "final": {name: directory / f"{source}_{name}.npy" for name in shapes},
        "partial": {name: directory / f"{source}_{name}.partial.npy" for name in shapes},
        "ids": directory / f"{source}_image_ids.json",
        "progress": directory / f"{source}_progress.json",
        "meta": directory / f"{source}_meta.json",
    }


def _region_check_array(path, shape, dtype):
    array = np.load(path, mmap_mode="r", allow_pickle=False)
    try:
        assert array.shape == shape and array.dtype == dtype, f"Wrong region array shape/dtype: {path}"
        assert path.stat().st_size == array.offset + int(np.prod(shape)) * dtype.itemsize, f"Incomplete array: {path}"
    finally:
        array._mmap.close()


def _region_source_state(data, cache_dir, source, config):
    ids = data["source_image_ids"][source]
    shapes = _region_shapes(len(ids), config["max_regions"])
    paths = _region_paths(cache_dir, source, shapes)
    key = _region_hash(config)
    if paths["ids"].exists():
        assert json.loads(paths["ids"].read_text(encoding="utf-8")) == ids, "Region image row order differs."
    if paths["meta"].exists():
        meta = json.loads(paths["meta"].read_text(encoding="utf-8"))
        if meta["cache_key"] != key or meta["config"] != config:
            raise ValueError("Incompatible completed region cache. Match its extraction settings or choose a new cache_dir.")
        assert meta["complete"]
        assert meta["source"] == source and meta["processed"] == len(ids) and paths["ids"].is_file()
        for name, (shape, dtype) in shapes.items():
            assert paths["final"][name].is_file() and not paths["partial"][name].exists()
            _region_check_array(paths["final"][name], shape, dtype)
            assert paths["final"][name].stat().st_size == meta["arrays"][name]["file_bytes"]
        return {"status": "complete", "processed": len(ids), "meta": meta, "paths": paths, "shapes": shapes}
    if paths["progress"].exists():
        progress = json.loads(paths["progress"].read_text(encoding="utf-8"))
        if progress["config"] != config or progress["cache_key"] != key:
            raise ValueError("Incompatible region checkpoint. Resume requires the same code, weights, settings, extraction batch and device.")
        assert progress["source"] == source
        assert paths["ids"].is_file() and progress["total"] == len(ids)
        offset = progress["processed"]
        assert isinstance(offset, int) and not isinstance(offset, bool) and 0 <= offset <= len(ids)
        assert np.isfinite(progress["elapsed_seconds"]) and progress["elapsed_seconds"] >= 0
        status = progress["status"]
        assert status in ["initializing", "running", "paused", "ready"]
        if status == "initializing":
            assert offset == 0
        if status == "ready":
            assert offset == len(ids)
        for name, (shape, dtype) in shapes.items():
            final, partial = paths["final"][name], paths["partial"][name]
            assert not (final.exists() and partial.exists()), f"Ambiguous final/partial files: {name}"
            if final.exists():
                assert status == "ready", "A final array without complete metadata requires a ready checkpoint."
                _region_check_array(final, shape, dtype)
            elif partial.exists():
                _region_check_array(partial, shape, dtype)
            elif status != "initializing":
                raise FileNotFoundError(f"Region checkpoint has no matching {name} array.")
        return {"status": status, "processed": offset, "progress": progress, "paths": paths, "shapes": shapes}
    if any(path.exists() for group in [paths["final"], paths["partial"]] for path in group.values()):
        raise ValueError("Region arrays exist without a compatible progress/metadata file. Review them or choose a new cache directory.")
    return {"status": "new", "processed": 0, "paths": paths, "shapes": shapes}


def _region_cache_config(data, detector, max_regions, score_threshold, nms_iou, batch_size, producer_code_id):
    if not isinstance(max_regions, int) or isinstance(max_regions, bool) or not 1 <= max_regions <= 255:
        raise ValueError("max_regions must be in 1..255 for uint8 counts.")
    if not 0 <= score_threshold <= 1 or not 0 <= nms_iou <= 1:
        raise ValueError("Score threshold and NMS IoU must be in [0,1].")
    identity = getattr(detector, "region_feature_metadata", {})
    digest = identity.get("weights_sha256", "")
    assert len(digest) == 64 and all(character in "0123456789abcdef" for character in digest)
    assert identity.get("weights") == REGION_WEIGHT_NAME and digest.startswith("258fb6c6")
    assert not detector.training and not any(parameter.requires_grad for parameter in detector.parameters())
    if not isinstance(producer_code_id, str) or len(producer_code_id) != 64 or any(character not in "0123456789abcdef" for character in producer_code_id):
        raise ValueError("Pass the protocol's 64-character producer_code_id fingerprint. Recompute it whenever the extraction source changes.")
    device = next(detector.parameters()).device
    device_info = {"device": str(device), "type": device.type,
                   "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
                   "compute_capability": list(torch.cuda.get_device_capability(device)) if device.type == "cuda" else None,
                   "cuda_runtime": torch.version.cuda if device.type == "cuda" else None}
    assert data["feature_sources"] == {"train": "train2017", "val": "val2017", "test": "train2017"}
    for source in REGION_SOURCES:
        ids = data["source_image_ids"][source]
        assert ids and ids == sorted(set(ids))
    # Use the ACTUAL call settings, not the builder's default metadata settings.
    return {
        **region_feature_contract(max_regions, score_threshold, nms_iou),
        "source_id": data["provenance"]["source_id"],
        "annotation_sha256": {source: data["provenance"]["annotations_sha256"][source] for source in REGION_SOURCES},
        "source_image_ids_sha256": {source: _region_hash(data["source_image_ids"][source]) for source in REGION_SOURCES},
        "weights_sha256": digest,
        "producer_torch": torch.__version__, "producer_torchvision": torchvision.__version__,
        "producer_code_id": producer_code_id, "extraction_batch_size": batch_size, "extraction_device": device_info,
        "array_dtypes": {name: str(dtype) for name, (_, dtype) in _region_shapes(1, max_regions).items()},
    }


def _region_additional_space(states):
    additional = 0
    for state in states.values():
        if state["status"] in ["complete", "ready"]:
            continue
        for name, (shape, dtype) in state["shapes"].items():
            expected = _region_file_bytes(shape, dtype)
            partial = state["paths"]["partial"][name]
            if not partial.exists():
                additional += expected
                continue
            info = partial.stat()
            if hasattr(info, "st_blocks"):
                additional += max(0, expected - info.st_blocks * 512)
            elif os.name == "nt" and hasattr(info, "st_file_attributes") and not info.st_file_attributes & (0x200 | 0x800):
                additional += max(0, expected - info.st_size)
            else:
                remaining_rows = shape[0] - state["processed"]
                additional += remaining_rows * int(np.prod(shape[1:])) * dtype.itemsize
    return additional


def _region_image_records(data, source, raw_image_root):
    annotation_path = Path(data["data_root"]) / "annotations" / f"captions_{source}.json"
    payload = annotation_path.read_bytes()
    assert hashlib.sha256(payload).hexdigest() == data["provenance"]["annotations_sha256"][source]
    annotation = json.loads(payload)
    images = {row["id"]: row for row in annotation["images"]}
    ids = data["source_image_ids"][source]
    assert sorted(images) == ids
    records = [images[image_id] for image_id in ids]
    directory = Path(raw_image_root) / source
    if not directory.is_dir():
        raise FileNotFoundError(f"Original photos missing: {directory}")
    with os.scandir(directory) as entries:
        available = {entry.name for entry in entries if entry.is_file()}
    for row in records:
        assert Path(row["file_name"]).name == row["file_name"]
    missing = [row["file_name"] for row in records if row["file_name"] not in available]
    if missing:
        raise FileNotFoundError(f"{source}: {len(missing)} photos missing; first {missing[:3]}")
    return records


def _region_pack_records(records, max_regions):
    assert records
    arrays = {}
    for name, (_, dtype) in _region_shapes(len(records), max_regions).items():
        values = [row[name] for row in records]
        if isinstance(values[0], torch.Tensor):
            values = torch.stack(values).detach().cpu().numpy()
        arrays[name] = np.asarray(values, dtype=dtype)
        assert arrays[name].shape == _region_shapes(len(records), max_regions)[name][0]
    _region_validate_chunk(arrays)
    for index, row in enumerate(records):
        count = row["num_regions"]
        height, width = row["original_size"]
        boxes = arrays["boxes"][index, :count]
        assert np.all(boxes[:, 0::2] <= width) and np.all(boxes[:, 1::2] <= height)
    return arrays


def _region_validate_chunk(arrays, score_threshold=None):
    features, masks, counts = arrays["features"], arrays["mask"], arrays["num_regions"]
    positions = np.arange(masks.shape[1])[None, :]
    expected_mask = positions < counts[:, None]
    assert np.all((counts >= 1) & (counts <= masks.shape[1]))
    assert np.array_equal(masks, expected_mask), "Mask/count or valid-region order mismatch."
    assert np.isfinite(features).all() and np.isfinite(arrays["boxes"]).all() and np.isfinite(arrays["scores"]).all()
    assert np.all(features[~masks] == 0) and np.all(arrays["boxes"][~masks] == 0)
    assert np.all(arrays["scores"][~masks] == 0) and np.all(arrays["labels"][~masks] == -1)
    boxes = arrays["boxes"][masks]
    assert np.all(boxes >= 0) and np.all(boxes[:, 2:] > boxes[:, :2])
    scores = arrays["scores"][masks]
    assert np.all((scores >= 0) & (scores <= 1))
    fallback = arrays["fallback"]
    assert np.all(counts[fallback] == 1)
    assert np.all(arrays["labels"][fallback, 0] == -1) and np.all(arrays["scores"][fallback, 0] == 0)
    normal_mask = masks & ~fallback[:, None]
    assert np.all(arrays["labels"][normal_mask] > 0)
    if score_threshold is not None:
        assert np.all(arrays["scores"][normal_mask] >= score_threshold)


def _region_validate_arrays(arrays, score_threshold, chunk_size=256):
    total = len(arrays["num_regions"])
    for start in range(0, total, chunk_size):
        _region_validate_chunk({name: array[start:start + chunk_size] for name, array in arrays.items()}, score_threshold)
    return {"all_finite": True, "counts_match_prefix_masks": True, "padding_valid": True,
            "images_checked": total, "valid_regions": int(arrays["num_regions"].sum(dtype=np.int64)),
            "count_min": int(arrays["num_regions"].min()), "count_mean": float(arrays["num_regions"].mean()),
            "count_max": int(arrays["num_regions"].max()),
            "fallback_images": int(arrays["fallback"].sum(dtype=np.int64)),
            "fallback_rate": float(arrays["fallback"].mean())}


def _region_finish_source(state, progress, config):
    paths = state["paths"]
    arrays = {name: np.load(paths["final"][name] if paths["final"][name].exists() else paths["partial"][name],
                            mmap_mode="r", allow_pickle=False) for name in state["shapes"]}
    validation_start = time.perf_counter()
    try:
        validation = _region_validate_arrays(arrays, config["score_threshold"])
    finally:
        for array in arrays.values():
            array._mmap.close()
    # A crash between renames is recoverable: the ready checkpoint permits a mix
    # of already-published final files and matching partial files, never both.
    for name in state["shapes"]:
        if paths["partial"][name].exists():
            paths["partial"][name].replace(paths["final"][name])
    array_metadata = {name: {"shape": list(shape), "dtype": str(dtype),
                             "file_bytes": paths["final"][name].stat().st_size,
                             "sha256": _region_file_sha256(paths["final"][name])}
                      for name, (shape, dtype) in state["shapes"].items()}
    elapsed = progress["elapsed_seconds"] + time.perf_counter() - validation_start
    meta = {**progress, "status": "complete", "complete": True, "elapsed_seconds": elapsed,
            "completed_utc": datetime.now(timezone.utc).isoformat(), "validation": validation,
            "arrays": array_metadata}
    _region_json(paths["meta"], meta)
    return meta


def _extract_region_source(data, detector, cache_dir, source, records, raw_image_root, config, batch_size):
    state = _region_source_state(data, cache_dir, source, config)
    if state["status"] == "complete":
        return state["meta"]
    if state["status"] == "ready":
        return _region_finish_source(state, state["progress"], config)
    paths = state["paths"]
    ids = data["source_image_ids"][source]
    offset = state["processed"]
    previous = state.get("progress", {})
    previous_seconds = float(previous.get("elapsed_seconds", 0))
    device = next(detector.parameters()).device
    common = {
        "schema_version": 1, "source": source, "total": len(ids), "config": config, "cache_key": _region_hash(config),
        "started_utc": previous.get("started_utc", datetime.now(timezone.utc).isoformat()),
        "device": str(device), "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
        "batch_size": batch_size, "resumed_from_row": offset,
        "timing": "Active allocation, image reading, detector/ROI computation, cache writes, full-array validation and SHA256 hashing; pauses and detector loading excluded.",
        "raw_image_validation": "EXIF-safe height/width pair checked against annotation dimensions; raw JPEG content bytes are not re-hashed.",
    }
    _region_json(paths["ids"], ids)
    if state["status"] == "new":
        _region_json(paths["progress"], {**common, "status": "initializing", "complete": False,
                                       "processed": 0, "elapsed_seconds": 0})
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    arrays = {}
    try:
        for name, (shape, dtype) in state["shapes"].items():
            path = paths["partial"][name]
            arrays[name] = np.load(path, mmap_mode="r+", allow_pickle=False) if path.exists() else np.lib.format.open_memmap(
                path, mode="w+", dtype=dtype, shape=shape)

        def save_progress(status):
            for array in arrays.values():
                array.flush()
            progress = {**common, "status": status, "complete": False, "processed": offset,
                        "elapsed_seconds": previous_seconds + time.perf_counter() - start,
                        "updated_utc": datetime.now(timezone.utc).isoformat()}
            _region_json(paths["progress"], progress)
            return progress

        try:
            save_progress("running")
            last_save = time.perf_counter()
            for first in range(offset, len(ids), batch_size):
                last = min(first + batch_size, len(ids))
                images = []
                for row in range(first, last):
                    item = records[row]
                    image = load_region_image(Path(raw_image_root) / source / item["file_name"])
                    if sorted(image.shape[-2:]) != sorted([item["height"], item["width"]]):
                        raise ValueError(f"Image {item['id']} dimensions {tuple(image.shape[-2:])} differ from COCO metadata. Use full-size original photos, not resized previews.")
                    images.append(image)
                results = extract_region_batch(detector, images, max_regions=config["max_regions"],
                                               score_threshold=config["score_threshold"], nms_iou=config["class_agnostic_nms_iou"])
                assert len(results) == last - first
                batch = _region_pack_records(results, config["max_regions"])
                for name in arrays:
                    arrays[name][first:last] = batch[name]
                offset = last  # Advance only after every array has received the whole batch.
                if time.perf_counter() - last_save >= 20 or offset == len(ids):
                    progress = save_progress("running")
                    print(f"Regions {source}: {offset:,}/{len(ids):,}, active {progress['elapsed_seconds']/60:.2f} min", flush=True)
                    last_save = time.perf_counter()
            assert offset == len(ids)
            progress = save_progress("ready")
        except BaseException:
            save_progress("paused")
            raise
    finally:
        for array in arrays.values():
            array._mmap.close()
    return _region_finish_source(state, progress, config)


def load_region_cache(data, cache_dir, expected_cache_key=None, expected_contract=None, producer_code_id=None):
    """Read compatible completed metadata/headers without loading all vectors."""
    first_meta = Path(cache_dir) / "train2017_meta.json"
    if not first_meta.is_file():
        raise FileNotFoundError("Region cache is not complete. Run its explicit extraction step first.")
    config = json.loads(first_meta.read_text(encoding="utf-8"))["config"]
    expected_contract = region_feature_contract() if expected_contract is None else expected_contract
    if any(config.get(key) != value for key, value in expected_contract.items()):
        raise ValueError("Completed region cache does not match the requested selection/transform contract.")
    if producer_code_id is not None and config.get("producer_code_id") != producer_code_id:
        raise ValueError("Completed region cache was created by a different extractor code fingerprint.")
    assert config["version"] == REGION_CACHE_VERSION and config["weights"] == REGION_WEIGHT_NAME
    assert config["source_id"] == data["provenance"]["source_id"]
    assert config["annotation_sha256"] == {source: data["provenance"]["annotations_sha256"][source] for source in REGION_SOURCES}
    assert config["source_image_ids_sha256"] == {source: _region_hash(data["source_image_ids"][source]) for source in REGION_SOURCES}
    if expected_cache_key is not None:
        assert _region_hash(config) == expected_cache_key, "Unexpected region-feature contract."
    files = {}
    for source in REGION_SOURCES:
        state = _region_source_state(data, cache_dir, source, config)
        if state["status"] != "complete":
            raise FileNotFoundError(f"Region {source} cache is {state['status']}; resume extraction first.")
        files[source] = {**{f"{name}_path": path for name, path in state["paths"]["final"].items()},
                         "image_ids_path": state["paths"]["ids"], "image_ids": data["source_image_ids"][source],
                         "cache_key": _region_hash(config), "shape": state["meta"]["arrays"]["features"]["shape"],
                         "meta": state["meta"]}
    return files


def extract_region_cache(data, detector, cache_dir, raw_image_root, batch_size=1,
                         max_regions=36, score_threshold=0.2, nms_iou=0.5, resume=True, reserve_gib=2,
                         producer_code_id=None):
    """Explicit cache step only. This does not train a decoder or evaluate caption metrics."""
    if not isinstance(batch_size, int) or isinstance(batch_size, bool) or batch_size < 1:
        raise ValueError("batch_size must be a positive integer; begin with 1.")
    if not np.isfinite(reserve_gib) or reserve_gib < 0:
        raise ValueError("reserve_gib must be nonnegative.")
    cache_dir = Path(cache_dir)
    config = _region_cache_config(data, detector, max_regions, score_threshold, nms_iou, batch_size, producer_code_id)
    contract = region_feature_contract(max_regions, score_threshold, nms_iou)
    states = {source: _region_source_state(data, cache_dir, source, config) for source in REGION_SOURCES}
    pending = [source for source, state in states.items() if state["status"] != "complete"]
    if not resume and any(states[source]["status"] != "new" for source in pending):
        raise ValueError("A matching incomplete cache exists. Set resume=True; it will not be replaced.")
    if not pending:
        return load_region_cache(data, cache_dir, _region_hash(config), contract, producer_code_id)
    active = [source for source in pending if states[source]["status"] != "ready"]
    records = {source: _region_image_records(data, source, raw_image_root) for source in active}
    existing_parent = cache_dir
    while not existing_parent.exists():
        existing_parent = existing_parent.parent
    additional = _region_additional_space(states)
    required = additional + int(reserve_gib * 1024 ** 3) if active else 0
    free = shutil.disk_usage(existing_parent).free
    print(f"Region cache: {additional/1024**3:.2f} GiB additional allocation; {required/1024**3:.2f} GiB with reserve; {free/1024**3:.2f} GiB free.", flush=True)
    if free < required:
        raise OSError("Not enough free space for the region cache and reserve.")
    cache_dir.mkdir(parents=True, exist_ok=True)
    for source in pending:
        _extract_region_source(data, detector, cache_dir, source, records.get(source), raw_image_root, config, batch_size)
    files = load_region_cache(data, cache_dir, _region_hash(config), contract, producer_code_id)
    report = {"complete": True, "config": config, "cache_key": _region_hash(config),
              "total_images": sum(len(item["image_ids"]) for item in files.values()),
              "total_active_seconds": sum(item["meta"]["elapsed_seconds"] for item in files.values()),
              "sources": {source: item["meta"] for source, item in files.items()}}
    _region_json(cache_dir / "region_cache_report.json", report)
    return files

In [ ]:
detector = None
if RUN_DETECTOR_SMOKE or EXTRACT_REGION_FEATURES:
    detector = build_frozen_region_detector(device,allow_download=ALLOW_WEIGHT_DOWNLOAD)
if RUN_DETECTOR_SMOKE:
    image_id = data["split_ids"]["train"][0]
    image_path = RAW_IMAGE_ROOT / "train2017" / f"{image_id:012d}.jpg"
    raw_image = load_region_image(image_path)
    sample = extract_region_batch(detector,[raw_image])[0]
    print("ROI features:",tuple(sample["features"].shape),"valid:",sample["num_regions"],"fallback:",sample["fallback"])
    plt.figure(figsize=(10,6))
    plt.imshow(raw_image.permute(1,2,0))
    for box in sample["boxes"][:sample["num_regions"]]:
        x1,y1,x2,y2 = box.tolist()
        plt.gca().add_patch(plt.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,color="red",linewidth=1))
    plt.axis("off")
    plt.show()
else:
    print("Real pretrained detector smoke test is off; run it on laptop B before extraction.")


In [ ]:
region_files = None
try:
    if EXTRACT_REGION_FEATURES:
        region_files = extract_region_cache(
            data,detector,CACHE_DIR,RAW_IMAGE_ROOT,batch_size=1,
            max_regions=36,score_threshold=0.2,nms_iou=0.5,resume=True,
            producer_code_id=protocol["region_producer_code_id"],
        )
    else:
        try:
            region_files = load_region_cache(
                data,CACHE_DIR,expected_contract=protocol["regions"],
                producer_code_id=protocol["region_producer_code_id"],
            )
        except FileNotFoundError:
            print("Region cache is missing or incomplete. Extraction remains off.")
    if region_files is not None:
        for source,entry in region_files.items():
            print(source,entry["shape"],entry["meta"]["validation"])
        print("Saved active extraction seconds:",sum(entry["meta"]["elapsed_seconds"] for entry in region_files.values()))
finally:
    # Release the detector before allocating the caption training model.
    detector = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 4. Batches and metrics

Training uses one random human caption per image per epoch: private RNG seeded42+epoch, sorted image IDs. Validation loss uses caption0. Keep all human captions for corpus metrics. Captions are32 content tokens maximum plus<start>/<end>; teacher input is captions[:,:-1], target is captions[:,1:]. Average loss over non-PAD target tokens.

Use Adam0.001, caption batch64, maximum100epochs, CIDEr-D early stopping patience3, gradient norm clipping5. AMP must unscale before clipping. Propagate the region mask in teacher forcing, preflight, evaluation and greedy generation. Greedy uses predicted previous tokens, forbids PAD/newSTART and stops atEND or33steps.

Metrics use **pycocoevalcap1.2**, identical project regex tokenization of predictions and original references, no UNK mapping of references. BLEU1/4 are corpus scores on0–1. CIDEr-D uses native scale, including its factor10 and no extra multiplication. This is not official leaderboard PTB tokenization.


In [ ]:
class RegionCaptionDataset(Dataset):
    def __init__(self, data, cache_files, split, seed=42):
        if split not in ["train", "val"]:
            raise ValueError("Use train or val. Test evaluation is outside this training stage.")
        self.data, self.split, self.seed = data, split, seed
        self.image_ids = data["split_ids"][split]
        self.features = self.visual_masks = self.num_regions = None
        source = data["feature_sources"][split]
        assert source in ["train2017", "val2017"]
        saved = cache_files[source]
        source_ids = json.loads(Path(saved["image_ids_path"]).read_text(encoding="utf-8"))
        assert source_ids == data["source_image_ids"][source], "Different cache row order."
        assert len(source_ids) == len(set(source_ids)), "Duplicate cache image IDs."
        self.feature_rows = {image_id: row for row, image_id in enumerate(source_ids)}
        assert all(image_id in self.feature_rows for image_id in self.image_ids)
        assert data["max_content_tokens"] == 32, "Keep the shared 32-content-token limit."
        try:
            self.features = np.load(saved["features_path"], mmap_mode="r", allow_pickle=False)
            self.visual_masks = np.load(saved["mask_path"], mmap_mode="r", allow_pickle=False)
            self.num_regions = np.load(saved["num_regions_path"], mmap_mode="r", allow_pickle=False)
            n = len(source_ids)
            assert self.features.shape == (n, 36, 1024) and self.features.dtype == np.float16
            assert self.visual_masks.shape == (n, 36) and self.visual_masks.dtype == np.bool_
            assert self.num_regions.shape == (n,) and self.num_regions.dtype == np.uint8
            assert np.all((self.num_regions >= 1) & (self.num_regions <= 36))
            assert np.array_equal(self.visual_masks.sum(axis=1), self.num_regions)
            self.set_epoch(0)
        except Exception:
            self.close()
            raise

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        rng = random.Random(self.seed + self.epoch)
        self.caption_choices = [
            rng.randrange(len(self.data["tokens"][self.split][image_id]))
            if self.split == "train" else 0
            for image_id in self.image_ids
        ]

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, index):
        image_id = self.image_ids[index]
        row = self.feature_rows[image_id]
        tokens = self.data["tokens"][self.split][image_id][self.caption_choices[index]]
        ids = self.data["ids"]
        caption = [ids["start"]] + [
            self.data["token_to_id"].get(token, ids["unk"]) for token in tokens[:32]
        ] + [ids["end"]]
        return {
            "features": torch.from_numpy(self.features[row].copy()).float(),
            "visual_mask": torch.from_numpy(self.visual_masks[row].copy()),
            "caption": torch.tensor(caption, dtype=torch.long),
            "image_id": image_id,
            "references": self.data["references"][self.split][image_id],
        }

    def close(self):
        for name in ["features", "visual_masks", "num_regions"]:
            array = getattr(self, name, None)
            if array is not None:
                mapping = getattr(array, "_mmap", None)
                if mapping is not None:
                    mapping.close()
                elif hasattr(array, "close"):
                    array.close()
                setattr(self, name, None)


def region_caption_collate(batch, pad_id):
    captions = [item["caption"] for item in batch]
    return {
        "features": torch.stack([item["features"] for item in batch]),
        "visual_mask": torch.stack([item["visual_mask"] for item in batch]),
        "captions": pad_sequence(captions, batch_first=True, padding_value=pad_id),
        "lengths": torch.tensor([len(caption) for caption in captions], dtype=torch.long),
        "image_ids": [item["image_id"] for item in batch],
        "references": [item["references"] for item in batch],
    }


def make_region_loader(dataset, batch_size=64, indices=None, seed=42, pin_memory=False):
    # Pass the shared shuffled index list for training; validation keeps source split order.
    selected = dataset if indices is None else Subset(dataset, indices)
    return DataLoader(
        selected, batch_size=batch_size, shuffle=False, num_workers=0,
        collate_fn=partial(region_caption_collate, pad_id=dataset.data["ids"]["pad"]),
        pin_memory=pin_memory, generator=torch.Generator().manual_seed(seed),
    )


def train_region_batch(model, batch, optimizer, scaler, device, pad_id, clip=5, amp=True):
    device = torch.device(device)
    use_amp = amp and device.type == "cuda"
    if use_amp and (scaler is None or not scaler.is_enabled()):
        raise ValueError("CUDA AMP requires an enabled GradScaler.")
    model.train()
    features = batch["features"].to(device)
    visual_mask = batch["visual_mask"].to(device)
    captions = batch["captions"].to(device)
    inputs, targets = captions[:, :-1], captions[:, 1:]
    token_count = int(targets.ne(pad_id).sum())
    if token_count == 0:
        raise ValueError("The batch has no non-padding target tokens.")
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
        logits = model(features, inputs, visual_mask=visual_mask)
        criterion = nn.CrossEntropyLoss(ignore_index=pad_id, reduction="sum")
        loss_sum = criterion(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1))
        loss = loss_sum / token_count
    if not torch.isfinite(loss):
        raise RuntimeError("Non-finite training loss.")
    if use_amp:
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
    # Epoch loss = total_loss_sum / total_token_count, not an average of batch averages.
    return float(loss_sum.detach()), token_count


@torch.no_grad()
def generate_region_batch(model, batch, device, ids, max_steps=33):
    return model.greedy(
        batch["features"].to(device), ids["start"], ids["end"], max_steps=max_steps,
        visual_mask=batch["visual_mask"].to(device),
    )

In [ ]:
class MetricEvaluator:
    def __init__(self, settings):
        version = package_metadata.version("pycocoevalcap")
        if version != "1.2":
            raise RuntimeError("Use pycocoevalcap==1.2 so all experiments use the same scorers.")
        self.token_pattern = settings["token_pattern"]
        re.compile(self.token_pattern)
        self.metadata = {
            "package": "pycocoevalcap",
            "version": version,
            "bleu": {
                "implementation": "Bleu(4)",
                "aggregation": "corpus",
                "scale": "0..1; no multiplication by 100",
            },
            "cider_d": {
                "implementation": "Cider(n=4,sigma=6.0)",
                "ngrams": 4,
                "sigma": 6.0,
                "scale": "native score, including the implementation's factor 10; no extra multiplication",
                "document_frequency": "all reference sets in the evaluated split",
            },
            "tokenization": {
                "name": "project_regex_v1",
                "normalization": "NFKC,lowercase,smart quotes,whitespace",
                "token_pattern": self.token_pattern,
                "reference_vocabulary_mapping": False,
                "note": "Same tokenizer for predictions and raw references; not COCO leaderboard PTB tokenization.",
            },
            "source_sha256": {
                name: hashlib.sha256(Path(inspect.getfile(cls)).read_bytes()).hexdigest()
                for name, cls in [
                    ("bleu", Bleu),
                    ("bleu_scorer", BleuScorer),
                    ("cider", Cider),
                    ("cider_scorer", CiderScorer),
                ]
            },
        }

    def tokenize(self, text):
        text = unicodedata.normalize("NFKC", text).lower()
        text = text.translate(
            str.maketrans(
                {
                    "\u2018": "'",
                    "\u2019": "'",
                    "\u02bc": "'",
                    "\u201c": '"',
                    "\u201d": '"',
                    "\u200b": " ",
                    "\ufeff": " ",
                }
            )
        )
        text = " ".join(text.split())
        return " ".join(re.findall(self.token_pattern, text))

    def score(self, records, expected_ids=None):
        # One corpus call is essential: averaging batch BLEU/CIDEr is incorrect.
        if len(records) < 2:
            raise ValueError("CIDEr-D needs a corpus with at least two images.")
        gts = {}
        res = {}
        for item in records:
            image_id = int(item["image_id"])
            if image_id in gts:
                raise ValueError(f"Duplicate prediction for image {image_id}")
            references = item["references"]
            if not isinstance(references, list) or not references:
                raise ValueError(f"Missing human references for image {image_id}")
            if not all(isinstance(text, str) and text.strip() for text in references):
                raise ValueError(f"Empty or invalid human reference for image {image_id}")
            if not isinstance(item["prediction"], str):
                raise TypeError("Each prediction must be a string.")
            gts[image_id] = [self.tokenize(text) for text in references]
            if not all(gts[image_id]):
                raise ValueError(f"A reference has no tokens for image {image_id}")
            res[image_id] = [self.tokenize(item["prediction"])]
        if expected_ids is not None:
            expected = [int(image_id) for image_id in expected_ids]
            if len(expected) != len(set(expected)) or set(gts) != set(expected):
                raise ValueError("Predictions do not cover exactly the requested evaluation split.")
        bleu, _ = Bleu(4).compute_score(gts, res, verbose=0)
        cider_d, _ = Cider(n=4, sigma=6.0).compute_score(gts, res)
        scores = {"bleu1": float(bleu[0]), "bleu4": float(bleu[3]), "cider_d": float(cider_d)}
        if not all(np.isfinite(value) for value in scores.values()):
            raise ValueError("The metric implementation returned a non-finite score.")
        return scores


def check_metrics(evaluator):
    # Distinct documents keep CIDEr's corpus IDF meaningful in this small check.
    sentences = [
        "A red bus drives down the road.",
        "Two cats sleep on the green sofa.",
        "A woman holds a yellow umbrella.",
    ]
    records = [
        {"image_id": index, "prediction": text, "references": [text, text]}
        for index, text in enumerate(sentences)
    ]
    exact = evaluator.score(records, expected_ids=range(len(records)))
    assert abs(exact["bleu1"] - 1) < 1e-6 and abs(exact["bleu4"] - 1) < 1e-6
    assert abs(exact["cider_d"] - 10) < 1e-6
    unrelated = [
        {**row, "prediction": "unrelated vocabulary never matches anything"} for row in records
    ]
    poor = evaluator.score(unrelated)
    assert poor["bleu1"] < 1e-6 and poor["bleu4"] < 1e-6 and poor["cider_d"] == 0
    empty = evaluator.score([{**row, "prediction": ""} for row in records])
    assert empty["cider_d"] == 0 and empty["bleu1"] < 1e-6
    # These are library corpus scores, not sentence scores averaged by our code.
    gts = {
        row["image_id"]: [evaluator.tokenize(text) for text in row["references"]] for row in records
    }
    res = {row["image_id"]: [evaluator.tokenize(row["prediction"])] for row in records}
    direct_bleu, _ = Bleu(4).compute_score(gts, res, verbose=0)
    direct_cider, _ = Cider(n=4, sigma=6.0).compute_score(gts, res)
    assert exact == {
        "bleu1": float(direct_bleu[0]),
        "bleu4": float(direct_bleu[3]),
        "cider_d": float(direct_cider),
    }
    try:
        evaluator.score(records, expected_ids=[0, 1, 99])
    except ValueError:
        pass
    else:
        raise AssertionError("Incomplete evaluation corpus was accepted.")
    print(
        "Metric checks passed: exact captions, wrong captions, empty predictions and full-corpus coverage."
    )
    return {"exact": exact, "unrelated": poor, "empty_predictions": empty}

In [ ]:
evaluator = MetricEvaluator(data["text_settings"])
assert evaluator.metadata == protocol["metrics"]
metric_checks = check_metrics(evaluator)
print("Metric implementation, source hashes, tokenizer and scales match stage3.")


## 5. Recipient integration checklist

The full training driver is intentionally assigned to laptop B's Codex. Read `PROMPT_FOR_CODEX_B.txt` and the code in `reference/SpatialAttention_reference.ipynb` without running that reference. Finish one self-contained **FasterRCNN_Stage5.ipynb** with:

- masked validation loss and greedy predictions covering all5000validation IDs/all25014raw references;
- real cached batch64 forward/backward/generation; masks, padded perturbation and tiny fixed4-image memorization with dropout0; then recreate the main model from seed42;
- best/last full checkpoints and plain weights, optimizer/scaler/RNG/config/provenance/history/patience/batch-progress, active timers and strict resume checks;
- one planned training run and validation summary, no test score or parameter search;
- executed notebook, all run artifacts and detector/cache metadata packaged for laptop A. Region cache can travel separately.

The CPU development reports shipped in verification/ are not a real pretrained-detector, full-cache or full-training result. Report separately what actually runs on laptop B.
